# Window-Length Audit

The previous preprocessing notebook proposed 10-second windows, but the representative Voisard trial has annotated walking bouts shorter than 10 seconds. This notebook audits the whole Voisard annotation set before the window length is fixed.

It uses metadata only, so it does not create signal tensors or train a model.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
manifest = pd.read_csv(PROJECT_ROOT / 'data' / 'interim' / 'ml_readiness_manifest.csv')
voisard = manifest[manifest['dataset_id'] == 'voisard_2025'].copy().reset_index(drop=True)
FS_HZ = 100.0
print('Voisard trials:', len(voisard))
print('Voisard participants:', voisard['subject'].nunique())


Voisard trials: 488
Voisard participants: 122


In [2]:
def walking_bounds(meta):
    uturn_start, uturn_end = meta['uturnBoundaries']
    events = []
    for event_name in ['leftGaitEvents', 'rightGaitEvents']:
        events.extend(meta.get(event_name) or [])
    pre = [event for event in events if event[1] < uturn_start]
    post = [event for event in events if event[0] > uturn_end]
    bounds = []
    if pre:
        bounds.append((min(event[0] for event in pre), max(event[1] for event in pre)))
    if post:
        bounds.append((min(event[0] for event in post), max(event[1] for event in post)))
    return sorted(bounds)


def count_windows(bounds, window_seconds, hop_fraction):
    window = int(window_seconds * FS_HZ)
    hop = int(window * hop_fraction)
    return sum(max(0, (end - start - window) // hop + 1) for start, end in bounds)


records = []
for _, row in voisard.iterrows():
    trial_dir = PROJECT_ROOT / row['trial_directory']
    meta = json.loads((trial_dir / f"{row['trial_id']}_meta.json").read_text(encoding='utf-8'))
    bounds = walking_bounds(meta)
    records.append({
        'dataset_id': row['dataset_id'],
        'participant_key': f"{row['dataset_id']}:{row['subject']}",
        'trial_id': row['trial_id'],
        'label': row['label'],
        'n_walking_segments': len(bounds),
        'walking_duration_s': sum(end - start for start, end in bounds) / FS_HZ,
        'longest_segment_s': max([end - start for start, end in bounds], default=0) / FS_HZ,
        'train_windows_5s': count_windows(bounds, 5, 0.5),
        'eval_windows_5s': count_windows(bounds, 5, 1.0),
        'train_windows_8s': count_windows(bounds, 8, 0.5),
        'eval_windows_8s': count_windows(bounds, 8, 1.0),
        'train_windows_10s': count_windows(bounds, 10, 0.5),
        'eval_windows_10s': count_windows(bounds, 10, 1.0),
    })

audit = pd.DataFrame(records)
audit.head()


,dataset_id,participant_key,trial_id,label,n_walking_segments,walking_duration_s,longest_segment_s,train_windows_5s,eval_windows_5s,train_windows_8s,eval_windows_8s,train_windows_10s,eval_windows_10s
0,voisard_2025,voisard_2025:HS_1,HS_1_1,healthy,2,12.15,6.16,2,2,0,0,0,0
1,voisard_2025,voisard_2025:HS_1,HS_1_2,healthy,2,11.55,6.03,2,2,0,0,0,0
2,voisard_2025,voisard_2025:HS_1,HS_1_3,healthy,2,10.92,5.47,2,2,0,0,0,0
3,voisard_2025,voisard_2025:HS_1,HS_1_4,healthy,2,11.46,5.85,2,2,0,0,0,0
4,voisard_2025,voisard_2025:HS_1,HS_1_5,healthy,2,11.51,5.95,2,2,0,0,0,0


In [3]:
summary_rows = []
for seconds in [5, 8, 10]:
    summary_rows.append({
        'window_seconds': seconds,
        'total_training_windows': int(audit[f'train_windows_{seconds}s'].sum()),
        'total_evaluation_windows': int(audit[f'eval_windows_{seconds}s'].sum()),
        'trials_with_training_windows': int((audit[f'train_windows_{seconds}s'] > 0).sum()),
        'trials_with_evaluation_windows': int((audit[f'eval_windows_{seconds}s'] > 0).sum()),
        'participants_with_training_windows': int(audit.loc[audit[f'train_windows_{seconds}s'] > 0, 'participant_key'].nunique()),
        'healthy_training_windows': int(audit.loc[audit['label'] == 'healthy', f'train_windows_{seconds}s'].sum()),
        'stroke_training_windows': int(audit.loc[audit['label'] == 'stroke', f'train_windows_{seconds}s'].sum()),
    })

decision = pd.DataFrame(summary_rows)
decision


,window_seconds,total_training_windows,total_evaluation_windows,trials_with_training_windows,trials_with_evaluation_windows,participants_with_training_windows,healthy_training_windows,stroke_training_windows
0,5,2145,1367,482,482,121,1039,1106
1,8,744,538,246,246,85,205,539
2,10,408,296,137,137,56,46,362


In [4]:
print('Walking-duration summary by label:')
print(audit.groupby('label')[['walking_duration_s', 'longest_segment_s']].describe().round(2))
print()
print('Window-length decision table:')
print(decision.to_string(index=False))
print()
print('Trials with no 5-second training window:')
print(audit.loc[audit['train_windows_5s'] == 0, ['trial_id', 'label', 'walking_duration_s', 'longest_segment_s']].to_string(index=False))


Walking-duration summary by label:
        walking_duration_s                                                   \
                     count   mean    std    min    25%    50%    75%    max   
label                                                                         
healthy              360.0  14.59   3.15   6.08  12.44  14.23  16.17  28.63   
stroke               128.0  28.96  15.08  13.94  20.32  24.21  31.81  91.70   

        longest_segment_s                                                 
                    count   mean   std   min    25%    50%    75%    max  
label                                                                     
healthy             360.0   7.65  1.68  3.29   6.47   7.38   8.49  14.56  
stroke              128.0  15.33  7.99  7.00  10.63  12.64  17.25  49.65  

Window-length decision table:
 window_seconds  total_training_windows  total_evaluation_windows  trials_with_training_windows  trials_with_evaluation_windows  participants_with_training_windows

In [5]:
interim = PROJECT_ROOT / 'data' / 'interim'
audit_path = interim / 'voisard_window_length_audit.csv'
decision_path = interim / 'window_length_decision.csv'
audit.to_csv(audit_path, index=False)
decision.to_csv(decision_path, index=False)
print(audit_path)
print(decision_path)


C:\Users\frank\Documents\MR-ICT Review Paper\data\interim\voisard_window_length_audit.csv
C:\Users\frank\Documents\MR-ICT Review Paper\data\interim\window_length_decision.csv


## Gate

Choose the shortest window that retains the majority of annotated Voisard trials and participants while still covering several gait cycles. The decision must be recorded in the next preprocessing/materialization notebook and applied consistently to both datasets.